In [1]:
pip install scikit-learn


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# -------------------------------------------------
# 1. Load and prepare data
# -------------------------------------------------
data = load_breast_cancer()
X = data.data          # 569 x 30
y = data.target       # 0/1 labels

scaler = StandardScaler()
X = scaler.fit_transform(X)

# -------------------------------------------------
# 2. Activation functions
# -------------------------------------------------
def threshold(z):
    return (z >= 0).astype(int)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# -------------------------------------------------
# 3. One Neuron Perceptron
# -------------------------------------------------
class OneNeuron:
    def __init__(self, activation="threshold", lr=0.01, epochs=1000):
        self.activation_name = activation
        self.lr = lr
        self.epochs = epochs

    def activate(self, z):
        if self.activation_name == "threshold":
            return threshold(z)
        else:
            return sigmoid(z)

    def fit(self, X, y):
        n, d = X.shape
        self.w = np.zeros(d)
        self.b = 0

        for _ in range(self.epochs):
            z = X @ self.w + self.b
            y_hat = self.activate(z)

            error = y_hat - y
            self.w -= self.lr * (X.T @ error) / n
            self.b -= self.lr * error.mean()

    def predict(self, X):
        z = X @ self.w + self.b
        y_hat = self.activate(z)
        return (y_hat >= 0.5).astype(int)

# -------------------------------------------------
# 4. Neural Network (1 hidden layer, 12 neurons)
# -------------------------------------------------
class SimpleNN:
    def __init__(self, n_hidden=12, lr=0.01, epochs=2000):
        self.n_hidden = n_hidden
        self.lr = lr
        self.epochs = epochs

    def fit(self, X, y):
        n, d = X.shape

        # weights
        self.W1 = np.random.randn(d, self.n_hidden) * 0.1
        self.b1 = np.zeros(self.n_hidden)
        self.W2 = np.random.randn(self.n_hidden) * 0.1
        self.b2 = 0

        for _ in range(self.epochs):
            # -------- forward --------
            z1 = X @ self.W1 + self.b1
            a1 = sigmoid(z1)

            z2 = a1 @ self.W2 + self.b2
            y_hat = sigmoid(z2)

            # -------- backward --------
            dz2 = y_hat - y
            dW2 = a1.T @ dz2 / n
            db2 = dz2.mean()

            da1 = np.outer(dz2, self.W2)
            dz1 = da1 * a1 * (1 - a1)
            dW1 = X.T @ dz1 / n
            db1 = dz1.mean(axis=0)

            # -------- update --------
            self.W2 -= self.lr * dW2
            self.b2 -= self.lr * db2
            self.W1 -= self.lr * dW1
            self.b1 -= self.lr * db1

    def predict(self, X):
        a1 = sigmoid(X @ self.W1 + self.b1)
        y_hat = sigmoid(a1 @ self.W2 + self.b2)
        return (y_hat >= 0.5).astype(int)

# -------------------------------------------------
# 5. Experiments with different train sizes
# -------------------------------------------------
train_sizes = [0.6, 0.7, 0.8]

for size in train_sizes:
    print("\n======================================")
    print(f" Train size: {int(size*100)}%")
    print("======================================")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, train_size=size, random_state=42
    )

    # -------- Perceptron (Threshold) --------
    p_thresh = OneNeuron("threshold", lr=0.01, epochs=1000)
    p_thresh.fit(X_train, y_train)

    y_train_pred = p_thresh.predict(X_train)
    y_test_pred = p_thresh.predict(X_test)

    print("\nPerceptron - Threshold")
    print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
    print("Test  Accuracy:", accuracy_score(y_test, y_test_pred))

    # -------- Perceptron (Sigmoid) --------
    p_sig = OneNeuron("sigmoid", lr=0.01, epochs=1000)
    p_sig.fit(X_train, y_train)

    y_train_pred = p_sig.predict(X_train)
    y_test_pred = p_sig.predict(X_test)

    print("\nPerceptron - Sigmoid")
    print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
    print("Test  Accuracy:", accuracy_score(y_test, y_test_pred))

    # -------- Neural Network --------
    nn = SimpleNN(n_hidden=12, lr=0.01, epochs=2000)
    nn.fit(X_train, y_train)

    y_train_pred = nn.predict(X_train)
    y_test_pred = nn.predict(X_test)

    print("\nNeural Network (12 hidden, sigmoid)")
    print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
    print("Test  Accuracy:", accuracy_score(y_test, y_test_pred))



 Train size: 60%

Perceptron - Threshold
Train Accuracy: 1.0
Test  Accuracy: 0.9385964912280702

Perceptron - Sigmoid
Train Accuracy: 0.9794721407624634
Test  Accuracy: 0.9868421052631579

Neural Network (12 hidden, sigmoid)
Train Accuracy: 0.9618768328445748
Test  Accuracy: 0.9780701754385965

 Train size: 70%

Perceptron - Threshold
Train Accuracy: 1.0
Test  Accuracy: 0.9649122807017544

Perceptron - Sigmoid
Train Accuracy: 0.9773869346733668
Test  Accuracy: 0.9883040935672515

Neural Network (12 hidden, sigmoid)
Train Accuracy: 0.957286432160804
Test  Accuracy: 0.9532163742690059

 Train size: 80%

Perceptron - Threshold
Train Accuracy: 1.0
Test  Accuracy: 0.9473684210526315

Perceptron - Sigmoid
Train Accuracy: 0.9824175824175824
Test  Accuracy: 0.9912280701754386

Neural Network (12 hidden, sigmoid)
Train Accuracy: 0.9736263736263736
Test  Accuracy: 0.9736842105263158
